In [0]:
import os
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

import mlflow
import mlflow.pytorch
from sklearn.metrics import confusion_matrix, classification_report

# -------------------------
# Config
# -------------------------
DATA_ROOT = "/Volumes/tomato_data/default/raw/tomato/"
TRAIN_DIR = os.path.join(DATA_ROOT, "train")
VAL_DIR   = os.path.join(DATA_ROOT, "val")
TEST_DIR  = os.path.join(DATA_ROOT, "test")

IMAGE_SIZE   = 224
BATCH_SIZE   = 32
NUM_EPOCHS   = 5
LEARNING_RATE = 1e-3
NUM_WORKERS  = 2
SEED         = 42
MODEL_NAME   = "tomato_disease_classifier"

CLASS_DISPLAY_NAMES = {
    "Tomato___Bacterial_spot": "Bacterial Spot",
    "Tomato___Early_blight": "Early Blight",
    "Tomato___Late_blight": "Late Blight",
    "Tomato___Leaf_Mold": "Leaf Mold",
    "Tomato___Septoria_leaf_spot": "Septoria Leaf Spot",
    "Tomato___healthy": "Healthy",
}

# -------------------------
# Reproducibility
# -------------------------
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print(f"Torch version: {torch.__version__}")

In [0]:
# ImageNet normalization stats — required because we use pretrained weights
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

eval_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

print("Transforms defined.")

In [0]:
# Load data
train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transforms)
val_dataset   = datasets.ImageFolder(VAL_DIR,   transform=eval_transforms)
test_dataset  = datasets.ImageFolder(TEST_DIR,  transform=eval_transforms)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

# ImageFolder assigns indices alphabetically — keep the mapping
CLASS_NAMES = train_dataset.classes
CLASS_TO_IDX = train_dataset.class_to_idx
NUM_CLASSES = len(CLASS_NAMES)

print(f"Classes ({NUM_CLASSES}): {CLASS_NAMES}")
print(f"Class → index: {CLASS_TO_IDX}")
print(f"Train batches: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}")

In [0]:
# create check points
def build_model(num_classes, freeze_backbone=True):
    model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)

    if freeze_backbone:
        for param in model.features.parameters():
            param.requires_grad = False

    # Replace classifier head
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, num_classes)
    return model

model = build_model(NUM_CLASSES, freeze_backbone=True).to(DEVICE)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

In [0]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")


CHECKPOINT_DIR = "/Volumes/tomato_data/default/raw/tomato/checkpoints/"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
BEST_MODEL_PATH = os.path.join(CHECKPOINT_DIR, "best_model.pt")


# Train / eval loops

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = outputs.max(1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)
        _, preds = outputs.max(1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    return (
        running_loss / total,
        correct / total,
        np.array(all_preds),
        np.array(all_labels),
    )



# DataLoaders (num_workers=0 for serverless compute, pin_memory=False for CPU)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=False
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=False
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=False
)


# MLflow experiment

mlflow.set_experiment("/Users/touhid9xx@gmail.com/tomato_classifier")

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LEARNING_RATE,
)

best_val_acc = 0.0
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

with mlflow.start_run(run_name="tomato_disease_mobilenetv2_transfer") as run:
    mlflow.log_params({
        "backbone": "mobilenet_v2",
        "pretrained": "IMAGENET1K_V1",
        "freeze_backbone": True,
        "image_size": IMAGE_SIZE,
        "batch_size": BATCH_SIZE,
        "epochs": NUM_EPOCHS,
        "learning_rate": LEARNING_RATE,
        "optimizer": "Adam",
        "loss": "CrossEntropyLoss",
        "num_classes": NUM_CLASSES,
        "seed": SEED,
    })

    for epoch in range(1, NUM_EPOCHS + 1):
        train_loss, train_acc = train_one_epoch(
            model, train_loader, criterion, optimizer, DEVICE
        )
        val_loss, val_acc, _, _ = evaluate(
            model, val_loader, criterion, DEVICE
        )

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        mlflow.log_metrics({
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": val_loss,
            "val_acc": val_acc,
        }, step=epoch)

        print(
            f"Epoch {epoch}/{NUM_EPOCHS} | "
            f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
            f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}"
        )

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(
                {
                    "model_state_dict": model.state_dict(),
                    "val_acc": best_val_acc,
                    "epoch": epoch,
                    "class_names": getattr(train_dataset, "classes", None),
                },
                BEST_MODEL_PATH,
            )
            print(f"  -> New best model saved to {BEST_MODEL_PATH}")

    print(f"\nBest val accuracy: {best_val_acc:.4f}")
    mlflow.log_metric("best_val_acc", best_val_acc)

    if os.path.exists(BEST_MODEL_PATH):
        mlflow.log_artifact(BEST_MODEL_PATH, artifact_path="checkpoints")

    RUN_ID = run.info.run_id

print(f"MLflow run ID: {RUN_ID}")
print(f"Best model checkpoint: {BEST_MODEL_PATH}")

In [0]:

CHECKPOINT_DIR = "/Volumes/tomato_data/default/raw/tomato/checkpoints/"
BEST_MODEL_PATH = os.path.join(CHECKPOINT_DIR, "best_model.pt")
CM_PATH = os.path.join(CHECKPOINT_DIR, "confusion_matrix.png")


# Load best checkpoint

ckpt = torch.load(BEST_MODEL_PATH, map_location=DEVICE)
model.load_state_dict(ckpt["model_state_dict"])
model.to(DEVICE)
model.eval()

print(f"Loaded checkpoint from epoch {ckpt.get('epoch', '?')} "
      f"(val_acc={ckpt.get('val_acc', float('nan')):.4f})")


# Evaluate on the test set

test_loss, test_acc, preds, labels = evaluate(
    model, test_loader, criterion, DEVICE
)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_acc:.4f}")


# Classification report

display_names = [CLASS_DISPLAY_NAMES.get(c, c) for c in CLASS_NAMES]

report = classification_report(
    labels, preds,
    target_names=display_names,
    digits=4,
)
print("\nClassification report:")
print(report)


# Confusion matrix plot

cm = confusion_matrix(labels, preds)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=display_names,
    yticklabels=display_names,
    ax=ax,
)
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title(f"Confusion Matrix — Test Accuracy {test_acc:.3f}")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(CM_PATH, dpi=120, bbox_inches="tight")
plt.show()

print(f"Confusion matrix saved to {CM_PATH}")

In [0]:
from mlflow.models.signature import infer_signature


# Write report + class mapping to the checkpoint volume

REPORT_PATH = os.path.join(CHECKPOINT_DIR, "classification_report.txt")
MAPPING_PATH = os.path.join(CHECKPOINT_DIR, "class_mapping.json")

with open(REPORT_PATH, "w") as f:
    f.write(report)

with open(MAPPING_PATH, "w") as f:
    json.dump({
        "class_names": list(CLASS_NAMES),
        "class_to_idx": CLASS_TO_IDX,
        "display_names": CLASS_DISPLAY_NAMES,
    }, f, indent=2)

print(f"Wrote {REPORT_PATH}")
print(f"Wrote {MAPPING_PATH}")


# Log artifacts to the same run and register the model

with mlflow.start_run(run_id=RUN_ID):
    mlflow.log_artifact(CM_PATH)
    mlflow.log_artifact(REPORT_PATH)
    mlflow.log_artifact(MAPPING_PATH)

    # Build a signature from a real forward pass so the registered
    # model carries proper input/output schema.
    model.eval()
    sample_input = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE).to(DEVICE)

    with torch.no_grad():
        sample_output = model(sample_input)

    signature = infer_signature(
        sample_input.cpu().numpy(),
        sample_output.cpu().numpy(),
    )

    mlflow.pytorch.log_model(
        pytorch_model=model,
        name="model",
        registered_model_name=MODEL_NAME,
        signature=signature,
        input_example=sample_input.cpu().numpy(),
    )

print(f"Model registered as '{MODEL_NAME}' from run {RUN_ID}")

In [0]:

TEST_DIR = "/Volumes/tomato_data/default/raw/tomato/test/"
SAMPLE_PRED_PATH = os.path.join(CHECKPOINT_DIR, "sample_prediction.png")

# Pick a sample image using dbutils.fs (reliable on UC volumes)
test_class_dir = TEST_DIR + CLASS_NAMES[0] + "/"
print(f"Listing: {test_class_dir}")

entries = dbutils.fs.ls(test_class_dir)
print(f"Entries found: {len(entries)}")
for e in entries[:3]:
    print(f"  {e.name}  ({e.size} bytes)")

IMG_EXTS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")
candidates = [e.path for e in entries
              if e.name.lower().endswith(IMG_EXTS)]

if not candidates:
    raise FileNotFoundError(f"No images found in {test_class_dir}")

sample_img_path = candidates[0].replace("dbfs:", "")
print(f"Sample image: {sample_img_path}")

# Predict
# ------------------------------------------------------------------
with Image.open(sample_img_path) as im:
    rgb = im.convert("RGB")
    img_tensor = eval_transforms(rgb).unsqueeze(0).to(DEVICE)
    display_img = rgb.copy()

model.eval()
with torch.no_grad():
    logits = model(img_tensor)
    probs = torch.softmax(logits, dim=1)[0].cpu().numpy()

pred_idx = int(np.argmax(probs))
pred_class = CLASS_NAMES[pred_idx]
pred_display = CLASS_DISPLAY_NAMES.get(pred_class, pred_class)
true_display = CLASS_DISPLAY_NAMES.get(CLASS_NAMES[0], CLASS_NAMES[0])

# ------------------------------------------------------------------
# Plot
# ------------------------------------------------------------------
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].imshow(display_img)
ax[0].axis("off")
ax[0].set_title(f"True: {true_display}")

ax[1].barh(
    [CLASS_DISPLAY_NAMES.get(c, c) for c in CLASS_NAMES],
    probs,
)
ax[1].set_xlim(0, 1)
ax[1].set_xlabel("Probability")
ax[1].set_title(f"Predicted: {pred_display} ({probs[pred_idx]:.3f})")
plt.tight_layout()
plt.savefig(SAMPLE_PRED_PATH, dpi=120, bbox_inches="tight")
plt.show()

# ------------------------------------------------------------------
# Log to the run
# ------------------------------------------------------------------
with mlflow.start_run(run_id=RUN_ID):
    mlflow.log_artifact(SAMPLE_PRED_PATH)

print(f"\nTrue: {true_display}")
print(f"Predicted: {pred_display} ({probs[pred_idx]:.3f})")